In [ ]:
!pip install "audio-separator[gpu]"
!apt-get install -y ffmpeg

In [ ]:
from google.colab import drive
import shutil 
import os
import subprocess
from audio_separator.separator import Separator

drive.flush_and_unmount()
drive.mount('/content/drive')

import torch
print(torch.cuda.is_available())

Mounted at /content/drive


In [20]:
def separate(file_path):
    separator = Separator(output_format='WAV')
    separator.load_model()

    output_audio = separator.separate(file_path)

    # save vocals for use for lyrics.ipynb
    output_folder = "/content/drive/MyDrive/karaoke_output"
    os.makedirs(output_folder, exist_ok=True)  # make sure folder exists
    destination = os.path.join(output_folder, os.path.basename(output_audio[1]))
    shutil.copy(output_audio[1], destination)
    print("Saved file to Drive:", destination)

    
    return output_audio # List: [0]: instrumental, [1]: vocals 

In [ ]:
def extract_audio(input_video): # If video, extract wav
    input_audio = "input.wav"
    subprocess.run([
        "ffmpeg",
        "-y",
        "-i", input_video,
        "-vn", 
        "-acodec", "pcm_s16le",  # WAV format
        input_audio
    ], check=True)
    return input_audio

In [22]:
def stitch_audio(input_video, output_audio, output_folder):
    file_name = os.path.splitext(os.path.basename(input_video))[0]
    output_video = os.path.join(output_folder, f"{file_name}_karaoke.mp4")
    
    subprocess.run([
    "ffmpeg",
    "-i", input_video,
    "-i", output_audio,   
    "-map", "0:v",  # video from first input
    "-map", "1:a",  # audio from second input
    "-c:v", "copy",  # copy video (no re-encoding)
    "-shortest",  # match shortest duration
    output_video
    ])

In [ ]:
!ffmpeg -i /content/drive/MyDrive/karaoke/videoplayback.mp4

In [ ]:
def conversion():
    input_video = "/content/drive/MyDrive/karaoke/videoplayback.mp4"
    output_folder="/content/drive/MyDrive/karaoke_output"
    input_audio = extract_audio(input_video)
    output_audio = separate(input_audio)
    print(output_audio)
    stitch_audio(input_video, output_audio[0], output_folder) # output_audio is a list

conversion()

INFO:audio_separator.separator.separator:Separator version 0.41.1 instantiating with output_dir: None, output_format: WAV
INFO:audio_separator.separator.separator:Output directory not specified. Using current working directory.
INFO:audio_separator.separator.separator:Using model directory from model_file_dir parameter: /tmp/audio-separator-models/
INFO:audio_separator.separator.separator:Operating System: Linux #1 SMP Mon Feb  2 12:27:57 UTC 2026
INFO:audio_separator.separator.separator:System: Linux Node: d14283aa4231 Release: 6.6.113+ Machine: x86_64 Proc: x86_64
INFO:audio_separator.separator.separator:Python Version: 3.12.12
INFO:audio_separator.separator.separator:PyTorch Version: 2.10.0+cu128
INFO:audio_separator.separator.separator:FFmpeg installed: ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
INFO:audio_separator.separator.separator:ONNX Runtime GPU package installed with version: 1.24.3
INFO:audio_separator.separator.separator:CUDA is av

Saved file to Drive: /content/drive/MyDrive/karaoke_output/input_(Vocals)_model_bs_roformer_ep_317_sdr_12.wav
['input_(Instrumental)_model_bs_roformer_ep_317_sdr_12.wav', 'input_(Vocals)_model_bs_roformer_ep_317_sdr_12.wav']
Extracted: input.wav
